[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/21_generative_model_rl.ipynb)

# 21. RL for diffusion and flow generation

생성 모델 전체를 돌리지 않고 trajectory step, reward, log-ratio, reward weighting이 어떻게 연결되는지 작은 tensor로 확인한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Tiny denoising trajectory

정해진 update로 x_t trajectory를 만든다.


In [ ]:
x = torch.tensor([[1.0, -1.0]], device=device)
trajectory = [x.clone()]

for t in [1.0, 0.5, 0.0]:
    velocity = -0.2 * x + 0.1 * t
    x = x + 0.25 * velocity
    trajectory.append(x.clone())

print(torch.stack(trajectory))


In [ ]:
_ = profile_call("one generative step", lambda z: z+0.25*(-0.2*z+0.1), trajectory[0])


## 2. Terminal reward

마지막 sample에 scalar reward를 부여한다.


In [ ]:
target = torch.tensor([[0.0, 0.0]], device=device)
final = trajectory[-1]
reward = -((final-target)**2).sum(dim=-1)

print("final:", final)
print("reward:", reward)


In [ ]:
_ = profile_call("terminal reward", lambda z: -((z-target)**2).sum(-1), final)


## 3. Reward-weighted regression

같은 flow matching loss에 reward weight를 곱기는 가장 단순한 형태를 본다.


In [ ]:
pred_v = torch.tensor([[0.1,-0.2]], device=device, requires_grad=True)
target_v = torch.tensor([[-0.3,0.4]], device=device)
weight = reward.detach().exp().clamp(max=3.0)

loss = weight * F.mse_loss(pred_v, target_v, reduction="none").mean(dim=-1)
print("weight:", weight)
print("weighted loss:", loss)


In [ ]:
_ = profile_call("reward weighted MSE", lambda: weight*F.mse_loss(pred_v.detach(),target_v,reduction="none").mean(-1))


## 4. Diffusion-policy log-ratio toy

old/new Gaussian transition mean 차이에서 log-prob ratio를 만든다.


In [ ]:
x_next = torch.tensor([[0.8,-0.7]], device=device)
old_mean = torch.tensor([[0.75,-0.65]], device=device)
new_mean = torch.tensor([[0.82,-0.72]], device=device)
sigma = torch.tensor(0.1, device=device)

def gaussian_logp(x, mean, sigma):
    return -0.5 * (((x-mean)/sigma)**2 + 2*torch.log(sigma) + math.log(2*math.pi)).sum(-1)

old_logp = gaussian_logp(x_next, old_mean, sigma)
new_logp = gaussian_logp(x_next, new_mean, sigma)
ratio = (new_logp-old_logp).exp()

print("ratio:", ratio)


In [ ]:
def transition_ratio_once():
    new_ = gaussian_logp(x_next, new_mean, sigma)
    old_ = gaussian_logp(x_next, old_mean, sigma)
    return (new_ - old_).exp()

_ = profile_call("Gaussian transition log-ratio", transition_ratio_once)


## 5. PPO-style trajectory objective

reward advantage와 transition ratio를 clipping한다.


In [ ]:
adv = reward - reward.mean()
ratio = ratio.expand_as(reward)

objective = torch.minimum(
    ratio * adv,
    ratio.clamp(0.8,1.2) * adv,
)
print("objective:", objective)


In [ ]:
_ = profile_call("generative PPO objective", lambda: torch.minimum(ratio*adv,ratio.clamp(.8,1.2)*adv))


## 6. Flow policy deterministic view

확률 transition 없이 velocity field를 reward 방향으로 재가중하는 단순 view를 비교한다.


In [ ]:
x_t = torch.tensor([[1.0,-1.0]], device=device)
v = torch.tensor([[-0.4,0.3]], device=device)
reward_scale = torch.tensor([[1.5]], device=device)

guided_v = reward_scale * v
x_next = x_t + 0.2 * guided_v

print("guided velocity:", guided_v)
print("next:", x_next)


In [ ]:
_ = profile_call("reward-scaled flow step", lambda: x_t+0.2*reward_scale*v)


## References and provenance

**[21.1] DDPO**
- 출처: Black et al., Training Diffusion Models with Reinforcement Learning
- 이 노트북에서 가져온 부분: denoising transition as a policy step

**[21.2] Diffusion-DPO**
- 출처: preference optimization for diffusion models
- 이 노트북에서 가져온 부분: pairwise/reward optimization on diffusion trajectories

**[21.3] Flow-GRPO / flow-policy RL lineages**
- 출처: recent flow-model RL papers and open implementations
- 이 노트북에서 가져온 부분: reward/ratio optimization over flow trajectories
